In [10]:
import mlflow
import os
mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", "http://100.113.186.28:5000"))
mlflow.agno.autolog()
mlflow.set_experiment("agent-versioning-test")


from videodeepsearch.agent.team import build_video_search_team
from videodeepsearch.clients.storage.postgre.client import PostgresClient
from videodeepsearch.clients.storage.minio.client import MinioStorageClient
from videodeepsearch.clients.storage.qdrant.image_client import ImageQdrantClient
from videodeepsearch.clients.storage.qdrant.segment_client import SegmentQdrantClient
from videodeepsearch.clients.storage.qdrant.audio_client import AudioQdrantClient
from videodeepsearch.clients.storage.elasticsearch.client import ElasticsearchOCRClient
from videodeepsearch.clients.storage.elasticsearch.schema import ElasticsearchConfig
from videodeepsearch.clients.inference.client import QwenVLEmbeddingClient, MMBertClient, SpladeClient
from videodeepsearch.clients.inference.schema import QwenVLEmbeddingConfig, MMBertConfig, SpladeConfig


In [11]:
from agno.models.openrouter import OpenRouterResponses
from agno.db.postgres import AsyncPostgresDb


def setup_models(settings):
    llm_cfg = settings.llm_provider
    models = {}

    greeter_cfg = llm_cfg.agents.greeter
    models["greeter"] = OpenRouterResponses(
        id=greeter_cfg.model_id,
        api_key=os.getenv("OPENROUTER_API_KEY"),
    )

    orchestrator_cfg = llm_cfg.agents.orchestrator
    models["orchestrator"] = OpenRouterResponses(
        id=orchestrator_cfg.model_id,
        api_key=os.getenv("OPENROUTER_API_KEY"),
    )

    planning_cfg = llm_cfg.agents.planning
    models["planning"] = OpenRouterResponses(
        id=planning_cfg.model_id,
        api_key=os.getenv("OPENROUTER_API_KEY"),
    )

    models["llm_tool"] = models["planning"]
    models["summarizer"] = models["planning"]

    return models


async def init_clients(settings):
    clients = {}

    clients["postgres_client"] = PostgresClient(
        database_url=settings.storage.postgres.connection_url
    )

    clients["agno_db"] = AsyncPostgresDb(
        db_url=settings.storage.postgres.connection_url,
        create_schema=True,
    )

    clients["minio_client"] = MinioStorageClient(
        host=settings.storage.minio.host,
        port=settings.storage.minio.port,
        access_key=settings.storage.minio.access_key,
        secret_key=settings.storage.minio.secret_key,
        secure=settings.storage.minio.secure,
    )

    qdrant_cfg = settings.storage.qdrant
    clients["image_qdrant_client"] = ImageQdrantClient(
        host=qdrant_cfg.host,
        port=qdrant_cfg.port,
        collection_name=qdrant_cfg.collection_name,
        grpc_port=qdrant_cfg.grpc_port,
        prefer_grpc=qdrant_cfg.prefer_grpc,
    )
    clients["segment_qdrant_client"] = SegmentQdrantClient(
        host=qdrant_cfg.host,
        port=qdrant_cfg.port,
        collection_name=qdrant_cfg.collection_name,
        grpc_port=qdrant_cfg.grpc_port,
        prefer_grpc=qdrant_cfg.prefer_grpc,
    )
    clients["audio_qdrant_client"] = AudioQdrantClient(
        host=qdrant_cfg.host,
        port=qdrant_cfg.port,
        collection_name=qdrant_cfg.collection_name,
        grpc_port=qdrant_cfg.grpc_port,
        prefer_grpc=qdrant_cfg.prefer_grpc,
    )

    es_cfg = settings.storage.elasticsearch
    clients["es_ocr_client"] = ElasticsearchOCRClient(
        config=ElasticsearchConfig(
            host=es_cfg.host,
            port=es_cfg.port,
            user=es_cfg.user,
            password=es_cfg.password,
            use_ssl=es_cfg.use_ssl,
            verify_certs=es_cfg.verify_certs,
            index_name=es_cfg.index_name,
            request_timeout=es_cfg.request_timeout,
        )
    )
    await clients["es_ocr_client"].connect()

    arango_cfg = settings.storage.arangodb
    arango_client = ArangoClient(hosts=arango_cfg.host)
    clients["arango_db"] = arango_client.db(
        arango_cfg.database,
        username=arango_cfg.username,
        password=arango_cfg.password,
    )

    inf_cfg = settings.inference
    clients["qwenvl_client"] = QwenVLEmbeddingClient(
        config=QwenVLEmbeddingConfig(base_url=inf_cfg.qwenvl.base_url)
    )
    clients["mmbert_client"] = MMBertClient(
        config=MMBertConfig(
            base_url=inf_cfg.mmbert.base_url,
            model_name=inf_cfg.mmbert.model_name,
        )
    )
    clients["splade_client"] = SpladeClient(
        config=SpladeConfig(
            url=inf_cfg.splade.url,
            model_name=inf_cfg.splade.model_name,
            timeout=inf_cfg.splade.timeout,
            verbose=inf_cfg.splade.verbose,
            max_batch_size=inf_cfg.splade.max_batch_size,
        )
    )

    return clients

In [12]:
from videodeepsearch.core.settings import load_settings
from arango import ArangoClient

settings = load_settings()
models = setup_models(settings)
clients = await init_clients(settings)

team = build_video_search_team(
    session_id="test-versioning-session",
    user_id="test-user",
    list_video_ids=[
        "0e64f1c0da591ca67f07b7f9",
        "c98019fd17ff4420ea47eee7",
        "3636d10a2ad4787733c9700d",
        "c510fac771767405c891bf64",
        "92ba4b2e27f460945fded9e5",
        "533914541945c2060c128da3",
        "946330031ead69b21354d038",
        "3636d10a2ad4787733c9700d",
        "9b17f473300a5436f0a053be",
        "1e1d300356360ed84020821c",
        "02d242459a690605ee3a8ddf",
        "0f48acd4ac783dfbdee85468",
        "b1abb34af1bc67cb712d5ffb"
    ],
    models=models,
    worker_models={},
    db=clients["agno_db"],
    image_qdrant_client=clients["image_qdrant_client"],
    segment_qdrant_client=clients["segment_qdrant_client"],
    audio_qdrant_client=clients["audio_qdrant_client"],
    qwenvl_client=clients["qwenvl_client"],
    mmbert_client=clients["mmbert_client"],
    splade_client=clients["splade_client"],
    postgres_client=clients["postgres_client"],
    minio_client=clients["minio_client"],
    es_ocr_client=clients["es_ocr_client"],
    arango_db=clients["arango_db"],
)

2026-04-04 20:35:50.281 | INFO     | videodeepsearch.clients.storage.elasticsearch.client:connect:88 - [ElasticsearchOCRClient] Connected to http://localhost:9200
2026-04-04 20:35:50.282 | DEBUG    | videodeepsearch.agent.member.worker.tool_selector:register:62 - [ToolSelector] Registered toolkit factory: 'search'
2026-04-04 20:35:50.283 | DEBUG    | videodeepsearch.agent.member.worker.tool_selector:register:62 - [ToolSelector] Registered toolkit factory: 'utility'
2026-04-04 20:35:50.283 | DEBUG    | videodeepsearch.agent.member.worker.tool_selector:register:62 - [ToolSelector] Registered toolkit factory: 'video'


In [13]:
logged_models = mlflow.search_logged_models(
    filter_string="name LIKE 'video-search-team%'",
    order_by=[{"field_name": "creation_timestamp", "ascending": False}],
)

In [14]:
model_id = logged_models.iloc[0]['model_id']

In [15]:
lm = mlflow.get_logged_model(model_id)

In [16]:
lm

LoggedModel(artifact_location='mlflow-artifacts:/3/models/m-639181a064cb4949bab5bd381597f8f7/artifacts', creation_timestamp=1775309731131, experiment_id='3', last_updated_timestamp=1775309731159, model_id='m-639181a064cb4949bab5bd381597f8f7', model_type='', model_uri='models:/m-639181a064cb4949bab5bd381597f8f7', name='video-search-team-f3d7f860-test-v1-dirty', source_run_id='', status=<LoggedModelStatus.READY: 'READY'>, status_message='')

In [17]:
context = mlflow.set_active_model(model_id=model_id)



2026/04/04 20:35:50 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-639181a064cb4949bab5bd381597f8f7


In [20]:

from dotenv import load_dotenv
load_dotenv()

context = mlflow.set_active_model(model_id=model_id)

video_ids = [
    "0e64f1c0da591ca67f07b7f9",
    "c98019fd17ff4420ea47eee7",
    "3636d10a2ad4787733c9700d",
    "c510fac771767405c891bf64",
    "92ba4b2e27f460945fded9e5",
    "533914541945c2060c128da3",
    "946330031ead69b21354d038",
    "3636d10a2ad4787733c9700d",
    "9b17f473300a5436f0a053be",
    "1e1d300356360ed84020821c",
    "02d242459a690605ee3a8ddf",
    "0f48acd4ac783dfbdee85468",
    "b1abb34af1bc67cb712d5ffb"
]

with mlflow.start_run(run_name=f"run-with-{model_id[:8]}"):
    mlflow.log_param("version_model_id", model_id)
    mlflow.log_param("user_demand", "Hello how are you")
    mlflow.log_param("video_ids", str())

    stream = team.arun(
        input="Hello how are you",
        session_state={
            "list_video_ids": video_ids,
            "user_demand": "Hello how are you",
        },
        stream=True,
        stream_events=True
    )
    async for chunk in stream:
        continue

    # Log result metrics
    session_metrics = await team.aget_session_metrics()
    mlflow.log_metrics({
        "input_tokens": session_metrics.input_tokens,
        "output_tokens": session_metrics.output_tokens,
        "total_tokens": session_metrics.total_tokens,
    })

    if session_metrics.cost:
        mlflow.log_metric("cost", session_metrics.cost)



2026/04/04 20:39:01 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-639181a064cb4949bab5bd381597f8f7


🏃 View run run-with-m-639181 at: http://100.113.186.28:5000/#/experiments/3/runs/6d879e7cf7514832b2592c1488656470
🧪 View experiment at: http://100.113.186.28:5000/#/experiments/3


In [19]:
mlflow.__version__

'3.10.1'